# TP2 : descente de gradient, critères d'arrêt et choix du pas
**Cours :** Algorithmes d'optimisation, séance 6 · **Intervenant :** Serigne Faye

**VERSION ENSEIGNANT (corrigé)**

## Objectifs
- coder vous-même la descente de gradient à pas fixe ;
- **prévoir** le comportement de l'algorithme selon le pas, puis le vérifier ;
- coder **plusieurs critères d'arrêt** et comprendre ce que chacun garantit réellement (et ses pièges) ;
- coder une recherche linéaire (backtracking) et le pas optimal ;
- choisir et justifier une configuration pour atteindre une précision donnée.

## Consignes
Exécutez les cellules dans l'ordre. Complétez les cellules marquées **À COMPLÉTER**. Les cellules de test vous disent immédiatement si votre code est correct. Répondez aux questions dans les cellules de texte : elles comptent autant que le code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Fonction de travail
Sauf mention contraire, on minimise $f(x, y) = x^2 + 10y^2$, de gradient $(2x, 20y)^\top$ et de minimum $x^* = (0, 0)$, en partant de $x_0 = (1, 1)$.

In [ ]:
def f(p): return p[0]**2 + 10*p[1]**2
def grad(p): return np.array([2*p[0], 20*p[1]])
x0 = np.array([1.0, 1.0])
x_star = np.zeros(2)

## Partie 1 : la descente à pas fixe, écrite par vous
**Exercice 1.1.** Compléter la fonction `descente_pas_fixe`. Elle renvoie le dernier point, la liste des itérés et la liste des valeurs de $f$.

In [ ]:
def descente_pas_fixe(f, grad, x0, alpha, max_iter):
    x = np.array(x0, dtype=float)
    xs, fs = [x.copy()], [f(x)]
    for k in range(max_iter):
        x = x - alpha * grad(x)
        xs.append(x.copy()); fs.append(f(x))
    return x, np.array(xs), np.array(fs)

In [ ]:
x_fin, xs, fs = descente_pas_fixe(f, grad, x0, 0.05, 200)
print(x_fin)
assert np.linalg.norm(x_fin - x_star) < 1e-6, "La descente ne converge pas : vérifiez la mise à jour"
assert np.all(np.diff(fs) <= 1e-12), "f doit décroître à chaque itération pour ce pas"

# vérification avec l'exemple de la séance 5 : f(x) = (x-3)^2, x0 = 0, alpha = 0.1, 20 itérations
x1, _, _ = descente_pas_fixe(lambda p: (p[0] - 3)**2, lambda p: np.array([2*(p[0] - 3)]), [0.0], 0.1, 20)
print(x1)   # attendu : environ 2.9654
assert abs(x1[0] - 2.9654) < 1e-3
print("OK")

## Partie 2 : prévoir, puis vérifier
Pour $f(x, y) = x^2 + 10y^2$, la plus grande valeur propre de la hessienne $\mathrm{diag}(2, 20)$ est $\lambda_{\max} = 20$. D'après le cours, la descente à pas constant converge si $0 < \alpha < 2/\lambda_{\max}$.

**Exercice 2.1.** **Avant d'exécuter quoi que ce soit**, écrivez dans le dictionnaire `predictions` le comportement que vous attendez pour chaque pas (300 itérations) : `"converge"`, `"lent"`, `"oscille"` ou `"diverge"`.

In [ ]:
pas_liste = [0.001, 0.01, 0.05, 0.09, 0.1, 0.11]
predictions = {0.001: "lent", 0.01: "lent", 0.05: "converge", 0.09: "converge", 0.1: "oscille", 0.11: "diverge"}

La fonction `comportement` (fournie) lance la descente et classe le résultat. Comparez avec vos prévisions.

In [ ]:
def comportement(alpha, n=300):
    _, xs, _ = descente_pas_fixe(f, grad, x0, alpha, n)
    d = np.linalg.norm(xs - x_star, axis=1)          # distance au minimum
    if not np.isfinite(d[-1]) or d[-1] > 1e6: return "diverge"
    if d[-1] < 1e-6: return "converge"
    if d[-1] < 0.99 * d[-50]: return "lent"           # la distance diminue, mais lentement
    return "oscille"

print("   pas | prévu    | observé  | distance finale")
for a in pas_liste:
    obs = comportement(a)
    d = np.linalg.norm(descente_pas_fixe(f, grad, x0, a, 300)[0] - x_star)
    print(f"{a:6.3f} | {predictions[a]:8s} | {obs:8s} | {d:10.2e}   {'OK' if predictions[a] == obs else '<-- écart'}")

**Question 2.2.** Pour chaque écart entre votre prévision et le résultat, expliquez ce que vous aviez mal anticipé. Que se passe-t-il exactement pour $\alpha = 0{,}1$ ? (Indice : calculez le facteur par lequel la composante $y$ est multipliée à chaque itération, $1 - 20\alpha$.)

**Réponse :** Pour $\alpha = 0{,}1 = 2/\lambda_{\max}$ (valeur critique), la composante $y$ est multipliée par $1 - 20\alpha = -1$ : elle change de signe à chaque itération sans diminuer, donc la suite ne converge pas (elle oscille entre $y = 1$ et $y = -1$) alors que $x$ converge. Pour $\alpha = 0{,}11$, le facteur vaut $-1{,}2$ : $|y|$ augmente de 20 % à chaque itération, ce qui diverge. Pour $\alpha = 0{,}001$ et $0{,}01$, la composante $x$ est multipliée par $1 - 2\alpha = 0{,}998$ et $0{,}98$ : la convergence est correcte mais lente. Les pas $0{,}05$ et $0{,}09$ convergent vite car $|1 - 20\alpha| \leq 0{,}8$.

## Partie 3 : les critères d'arrêt
Jusqu'ici, on a lancé un nombre fixe d'itérations. En pratique, il faut décider **quand s'arrêter**. Quatre critères, avec un seuil `tol` :

| nom | l'algorithme s'arrête quand |
|---|---|
| `"grad"` | $\|\nabla f(x_{k+1})\| < \text{tol}$ |
| `"pas"` | $\|x_{k+1} - x_k\| < \text{tol}$ |
| `"fval"` | $\|f(x_{k+1}) - f(x_k)\| < \text{tol}$ |
| `"grad_rel"` | $\|\nabla f(x_{k+1})\| < \text{tol} \times \|\nabla f(x_0)\|$ (critère **relatif**) |

En plus, on garde toujours un nombre maximal d'itérations `max_iter` et une détection de divergence.

**Exercice 3.1.** Compléter `test_arret`, qui renvoie `True` quand l'algorithme doit s'arrêter.

In [ ]:
def test_arret(critere, tol, x_old, x_new, f_old, f_new, g_new, g0_norm):
    if critere == "grad":
        return np.linalg.norm(g_new) < tol
    if critere == "pas":
        return np.linalg.norm(x_new - x_old) < tol
    if critere == "fval":
        return abs(f_new - f_old) < tol
    if critere == "grad_rel":
        return np.linalg.norm(g_new) < tol * g0_norm
    return False

In [ ]:
g_petit, g_grand, z = np.array([1e-7, 0.0]), np.array([1.0, 0.0]), np.zeros(2)
assert test_arret("grad", 1e-6, z, z, 0, 0, g_petit, 1.0)
assert not test_arret("grad", 1e-6, z, z, 0, 0, g_grand, 1.0)
assert test_arret("pas", 1e-6, z, z + 1e-8, 0, 0, g_grand, 1.0)
assert not test_arret("pas", 1e-6, z, z + 1.0, 0, 0, g_grand, 1.0)
assert test_arret("fval", 1e-6, z, z, 1.0, 1.0 - 1e-9, g_grand, 1.0)
assert not test_arret("fval", 1e-6, z, z, 1.0, 0.5, g_grand, 1.0)
assert test_arret("grad_rel", 1e-6, z, z, 0, 0, g_petit, 1.0)
assert not test_arret("grad_rel", 1e-6, z, z, 0, 0, g_petit, 1e-3)
print("test_arret : OK")

La fonction `descente` (fournie) utilise votre `test_arret`. Le paramètre `pas` peut être un nombre ou une règle `regle(x, g)` qui renvoie le pas à chaque itération (on s'en servira en partie 4). Elle renvoie le dernier point et un dictionnaire avec le nombre d'itérations, le statut (`converge`, `max_iter` ou `divergence`) et l'historique.

In [ ]:
def descente(f, grad, x0, pas, max_iter=10000, tol=1e-6, critere="grad"):
    x = np.array(x0, dtype=float)
    g = grad(x); g0_norm = np.linalg.norm(g)
    xs, fs = [x.copy()], [f(x)]
    statut = "max_iter"
    if critere == "grad" and test_arret("grad", tol, x, x, f(x), f(x), g, g0_norm):
        return x, {"xs": np.array(xs), "fs": np.array(fs), "iters": 0, "statut": "converge"}
    for k in range(max_iter):
        t = pas(x, g) if callable(pas) else pas
        x_new = x - t * g
        if not np.all(np.isfinite(x_new)) or np.linalg.norm(x_new) > 1e8:
            statut = "divergence"; break
        g_new = grad(x_new)
        x_old, f_old, f_new = x, f(x), f(x_new)
        x, g = x_new, g_new
        xs.append(x.copy()); fs.append(f_new)
        if test_arret(critere, tol, x_old, x, f_old, f_new, g, g0_norm):
            statut = "converge"; break
    return x, {"xs": np.array(xs), "fs": np.array(fs), "iters": len(xs) - 1, "statut": statut}

### 3.2 Les quatre critères sur le même problème
**Exercice 3.2.** Avec $\alpha = 0{,}05$ et `tol = 1e-6`, lancer `descente` pour chacun des quatre critères et afficher le nombre d'itérations, le statut et la **précision réellement atteinte** $\|x - x^*\|$.

In [ ]:
print("critère  | itérations | statut   | précision réelle ‖x − x*‖")
for critere in ["grad", "pas", "fval", "grad_rel"]:
    x, h = descente(f, grad, x0, 0.05, tol=1e-6, critere=critere)
    print(f"{critere:8s} | {h['iters']:10d} | {h['statut']:8s} | {np.linalg.norm(x - x_star):.2e}")

**Question 3.3.** Le seuil `tol` est le même pour les quatre critères, pourtant la précision obtenue est très différente. Pourquoi ? Quel critère s'arrête le plus tôt, et que garantit-il réellement sur la distance au minimum ?

**Réponse :** Chaque critère mesure une quantité différente : `grad` mesure la pente ($\|\nabla f\| < 10^{-6}$ donne une distance au minimum d'environ $5 \times 10^{-7}$, car $\nabla f = 2x$ près du minimum) ; `pas` mesure la longueur du dernier déplacement $\alpha \|\nabla f\|$ (il s'arrête quand $\|\nabla f\| < 2 \times 10^{-5}$, soit une distance d'environ $10^{-5}$) ; `fval` s'arrête dès que $f$ varie de moins de $10^{-6}$ d'une itération à l'autre, ce qui arrive **loin** du minimum (distance $\approx 2 \times 10^{-3}$ ici : 59 itérations seulement) car $f$ est quadratique et ne varie lentement que parce que l'algorithme ralentit. Seul `grad` est directement lié à la condition d'optimalité $\nabla f = 0$.

### 3.4 Piège n°1 : un pas minuscule
**Exercice 3.4.** Lancer `descente` avec $\alpha = 10^{-9}$ (`max_iter=1000`, `tol=1e-6`) pour les critères `"grad"`, `"pas"` et `"fval"`. Afficher le statut, le nombre d'itérations et la distance au minimum.

In [ ]:
for critere in ["grad", "pas", "fval"]:
    x, h = descente(f, grad, x0, 1e-9, max_iter=1000, tol=1e-6, critere=critere)
    print(f"{critere:5s} : statut = {h['statut']:9s}, itérations = {h['iters']:5d}, ‖x − x*‖ = {np.linalg.norm(x - x_star):.3f}")

**Question 3.5.** Que déclarent `pas` et `fval` ? Est-ce vrai ? Que fait `grad` ? Quel est le danger pour quelqu'un qui n'afficherait que « l'algorithme a convergé » ?

**Réponse :** Les critères `pas` et `fval` déclarent la **convergence dès la première itération** alors que l'on est toujours en $(1, 1)$, à distance $1{,}41$ du minimum : un pas minuscule produit forcément un déplacement minuscule et une variation minuscule de $f$, sans que l'on soit près d'un minimum. C'est une **fausse convergence**. Le critère `grad` ne s'y laisse pas tromper : le gradient reste grand, l'algorithme va jusqu'à `max_iter` et le statut `max_iter` signale honnêtement qu'il n'a pas convergé. Un résultat « convergé » ne vaut donc que si le critère utilisé est pertinent.

### 3.6 Piège n°2 : une fonction à une autre échelle
On considère la même fonction multipliée par $10^{-8}$, avec le pas multiplié par $10^{8}$ (la dynamique de l'algorithme est identique).

In [ ]:
f_s = lambda p: 1e-8 * f(p)
grad_s = lambda p: 1e-8 * grad(p)
alpha_s = 0.05 / 1e-8

**Exercice 3.7.** Lancer `descente` sur `f_s` avec `tol = 1e-6` pour les critères `"grad"` et `"grad_rel"`. Comparer le nombre d'itérations et la précision atteinte à ce que vous avez obtenu en 3.2.

In [ ]:
for critere in ["grad", "grad_rel"]:
    x, h = descente(f_s, grad_s, x0, alpha_s, tol=1e-6, critere=critere)
    print(f"{critere:8s} : statut = {h['statut']:8s}, itérations = {h['iters']:4d}, ‖x − x*‖ = {np.linalg.norm(x - x_star):.2e}")

**Question 3.8.** Que se passe-t-il avec le critère absolu `grad` ? Pourquoi le critère relatif `grad_rel` est-il plus robuste ?

**Réponse :** Le critère absolu s'arrête **à l'itération 0** : le gradient initial vaut environ $2 \times 10^{-7}$, déjà inférieur à `tol`, donc l'algorithme déclare la convergence au point de départ, à distance $1{,}41$ du minimum. Le critère relatif compare le gradient à sa valeur initiale : il est invariant par changement d'échelle de $f$ et se comporte exactement comme dans le cas non mis à l'échelle (110 itérations, précision $\approx 10^{-5}$). Un seuil absolu n'a de sens que si l'on connaît l'ordre de grandeur de $\nabla f$.

### 3.9 Divergence et nombre maximal d'itérations
**Exercice 3.9.** Lancer `descente` avec $\alpha = 0{,}11$ et le critère `"grad"`. Quel statut obtient-on, et après combien d'itérations ?

In [ ]:
x, h = descente(f, grad, x0, 0.11, tol=1e-6, critere="grad")
print(h["statut"], h["iters"])

**Question 3.10.** Pourquoi faut-il toujours prévoir un `max_iter` et une détection de divergence, quel que soit le critère d'arrêt choisi ?

**Réponse :** Aucun critère ne s'arrête si l'algorithme diverge (le gradient grandit) ou s'il oscille sans converger : sans `max_iter`, la boucle serait infinie, et sans détection de divergence on calculerait des nombres de plus en plus grands jusqu'au dépassement de capacité. Le statut renvoyé (`divergence`, `max_iter`) permet de savoir si le résultat est fiable.

## Partie 4 : recherche linéaire (backtracking)
Plutôt que de fixer le pas, on le choisit à chaque itération : on part de $t = 1$ et on le divise par 2 tant que la condition d'Armijo n'est pas satisfaite :
$$f(x - t g) \leq f(x) - c\, t\, \|g\|^2, \qquad g = \nabla f(x), \quad c = 10^{-4}.$$

**Exercice 4.1.** Écrire `backtracking(f, grad, x)` qui renvoie le pas $t$.

In [ ]:
def backtracking(f, grad, x, t0=1.0, beta=0.5, c=1e-4):
    g = grad(x)
    t = t0
    while f(x - t * g) > f(x) - c * t * (g @ g):
        t *= beta
    return t

In [ ]:
t = backtracking(f, grad, x0)
g = grad(x0)
assert f(x0 - t * g) <= f(x0), "Le pas trouvé doit faire diminuer f"
assert 0 < t <= 1
print("pas initial choisi :", t)

**Exercice 4.2.** Comparer, avec le critère `"grad_rel"` (`tol = 1e-6`) : le pas fixe $0{,}05$, le pas fixe $0{,}09$ et le backtracking (`pas=lambda x, g: backtracking(f, grad, x)`). Afficher le nombre d'itérations et la précision.

In [ ]:
configs = {"fixe 0.05": 0.05, "fixe 0.09": 0.09, "backtracking": lambda x, g: backtracking(f, grad, x)}
for nom, pas in configs.items():
    x, h = descente(f, grad, x0, pas, tol=1e-6, critere="grad_rel")
    print(f"{nom:13s} : {h['iters']:4d} itérations, ‖x − x*‖ = {np.linalg.norm(x - x_star):.2e}")

**Question 4.3.** Que constatez-vous ? Quel est l'avantage du backtracking, et quel est son coût caché que le nombre d'itérations ne montre pas ?

**Réponse :** Le backtracking converge en environ 48 itérations, contre 63 pour le meilleur pas fixe testé ($0{,}09$) et 110 pour $0{,}05$ : il s'adapte à la fonction sans qu'on ait à connaître $\lambda_{\max}$ et ne diverge pas. Son coût caché est qu'il évalue $f$ plusieurs fois par itération (une à chaque réduction du pas), donc une itération est plus chère qu'avec un pas fixe.

## Partie 5 : pas optimal pour une fonction quadratique
Pour $f(x) = \tfrac12 x^\top A x$ avec $A = \mathrm{diag}(2, 20)$ (c'est notre fonction), la recherche linéaire exacte dans la direction $-g$ donne
$$t_k = \frac{g^\top g}{g^\top A g}.$$

**Exercice 5.1.** Écrire la règle `regle_optimale(x, g)`, puis comparer pas fixe $0{,}05$, pas optimal et backtracking depuis $x_0' = (1\,;\,0{,}1)$ avec le critère `"grad_rel"`. Tracer les trajectoires du pas optimal et du pas fixe sur les courbes de niveau.

In [ ]:
A = np.diag([2.0, 20.0])
def regle_optimale(x, g):
    return (g @ g) / (g @ A @ g)

xb = np.array([1.0, 0.1])
res = {}
for nom, pas in {"fixe 0.05": 0.05, "optimal": regle_optimale, "backtracking": lambda x, g: backtracking(f, grad, x)}.items():
    x, res[nom] = descente(f, grad, xb, pas, tol=1e-6, critere="grad_rel")
    print(f"{nom:13s} : {res[nom]['iters']:4d} itérations")

xs = np.linspace(-1.2, 1.2, 200); X, Y = np.meshgrid(xs, xs)
plt.figure(figsize=(6, 6)); plt.contour(X, Y, X**2 + 10*Y**2, levels=20, alpha=0.4)
plt.plot(res["optimal"]["xs"][:, 0], res["optimal"]["xs"][:, 1], "o-", ms=3, label="pas optimal")
plt.plot(res["fixe 0.05"]["xs"][:, 0], res["fixe 0.05"]["xs"][:, 1], "s-", ms=3, label="pas fixe 0.05")
plt.legend(); plt.show()

**Question 5.2.** Décrivez les trajectoires. Pourquoi le pas optimal « zigzague »-t-il dans la vallée ?

**Réponse :** Avec le pas optimal, on minimise exactement $f$ sur la droite de descente : le nouveau gradient est alors orthogonal au précédent, d'où un zigzag entre les deux flancs de la vallée. Ici : 69 itérations pour le pas optimal contre 128 pour le pas fixe $0{,}05$ (et 55 pour le backtracking). Plus la vallée est étroite, plus le zigzag est marqué.

## Partie 6 : le défi
On minimise $f_\kappa(x, y) = \tfrac12 (x^2 + 50 y^2)$ (conditionnement $\kappa = 50$) depuis $(1\,;\,0{,}02)$. **Objectif :** atteindre une distance au minimum **inférieure à $10^{-4}$** avec le moins d'itérations possible, en choisissant la règle de pas et le seuil `tol` (critère `"grad_rel"`).

La fonction `evalue` (fournie) renvoie le nombre d'itérations, la distance au minimum et le statut.

In [ ]:
kappa = 50
A_k = np.diag([1.0, kappa])
f_k = lambda p: 0.5 * p @ A_k @ p
grad_k = lambda p: A_k @ p
x_k0 = np.array([1.0, 1.0 / kappa])

def evalue(pas, tol):
    x, h = descente(f_k, grad_k, x_k0, pas, max_iter=20000, tol=tol, critere="grad_rel")
    return h["iters"], np.linalg.norm(x), h["statut"]

print(evalue(0.03, 1e-4))   # exemple d'appel

**Exercice 6.1.** Pour chaque configuration et chaque `tol` de la liste, relever le nombre d'itérations et la distance obtenue, puis repérer pour chaque configuration la **plus grande** valeur de `tol` qui atteint la précision demandée.

In [ ]:
configs = {
    "fixe 2/(1+kappa)": 2 / (1 + kappa),
    "fixe 0.03": 0.03,
    "backtracking": lambda x, g: backtracking(f_k, grad_k, x),
    "optimal": lambda x, g: (g @ g) / (g @ A_k @ g),
}
for nom, pas in configs.items():
    for tol in [1e-2, 1e-3, 1e-4, 1e-5]:
        it, d, st = evalue(pas, tol)
        print(f"{nom:17s} tol = {tol:.0e} : {it:4d} itérations, distance = {d:.1e}  {'✔' if d < 1e-4 else '✘'}")

**Question 6.2.** Quelle configuration (règle de pas et `tol`) choisissez-vous, et pourquoi ? Que remarquez-vous sur « fixe 2/(1+kappa) » et « optimal » ? Quel critère de choix avez-vous utilisé en plus du nombre d'itérations ?

**Réponse :** Pour atteindre une distance inférieure à $10^{-4}$, il faut `tol = 1e-4` pour la plupart des configurations (avec `tol = 1e-3`, la distance reste de l'ordre de $10^{-3}$). Le backtracking est le meilleur en nombre d'itérations (environ 208), devant le pas fixe $2/(1+\kappa)$ et le pas optimal (231 chacun). Le pas fixe $0{,}03$ n'atteint $10^{-4}$ qu'avec `tol = 1e-5` (environ 367 itérations), car sa précision pour `tol = 1e-4` est de $1{,}4 \times 10^{-4}$. Remarque : « fixe $2/(1+\kappa)$ » et « optimal » donnent exactement les mêmes résultats, car le point de départ $(1, 1/\kappa)$ est le cas le plus défavorable pour le pas optimal (il alterne alors entre deux valeurs égales à $2/(1+\kappa)$). Autre critère à discuter : le **coût par itération** (le backtracking évalue $f$ plusieurs fois par itération) et la robustesse (le pas fixe exige de connaître $\lambda_{\max}$).

## Partie 7 (bonus) : pour aller plus loin
### 7.1 Influence du conditionnement
**Exercice 7.1.** Pour $f_\kappa(x, y) = \tfrac12 (x^2 + \kappa y^2)$ avec $\kappa \in \{1, 10, 100\}$, depuis $(1\,;\,1/\kappa)$, comparer le nombre d'itérations (critère `"grad_rel"`, `tol=1e-6`) du pas fixe $2/(1+\kappa)$ et du backtracking.

In [ ]:
print(" kappa | pas fixe | backtracking")
for k in [1, 10, 100]:
    Ak = np.diag([1.0, k]); fk = lambda p: 0.5 * p @ Ak @ p; gk = lambda p: Ak @ p
    xk = np.array([1.0, 1.0 / k])
    _, h1 = descente(fk, gk, xk, 2 / (1 + k), max_iter=20000, tol=1e-6, critere="grad_rel")
    _, h2 = descente(fk, gk, xk, lambda x, g: backtracking(fk, gk, x), max_iter=20000, tol=1e-6, critere="grad_rel")
    print(f"{k:6d} | {h1['iters']:8d} | {h2['iters']:12d}")

**Question 7.2.** Comment le nombre d'itérations évolue-t-il avec $\kappa$ ?

**Réponse :** Il croît à peu près proportionnellement à $\kappa$ (de l'ordre de 70 itérations pour $\kappa = 10$ et 700 pour $\kappa = 100$) : un mauvais conditionnement (vallée étroite) ralentit fortement la descente de gradient. Les méthodes de Newton et quasi-Newton (séance 7) corrigent ce défaut.

### 7.2 La fonction de Rosenbrock
$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$, de minimum $(1, 1)$, depuis $(-1{,}2\,;\,1)$.

**Exercice 7.3.** Comparer le pas fixe $10^{-3}$ (`max_iter=20000`) et le backtracking (`max_iter=50000`), critère `"grad"`, `tol=1e-6`. Afficher statut, itérations et point obtenu.

In [ ]:
rosen = lambda p: (1 - p[0])**2 + 100 * (p[1] - p[0]**2)**2
rgrad = lambda p: np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0]**2), 200 * (p[1] - p[0]**2)])
r0 = np.array([-1.2, 1.0])
xa, ha = descente(rosen, rgrad, r0, 1e-3, max_iter=20000, critere="grad")
xb2, hb = descente(rosen, rgrad, r0, lambda x, g: backtracking(rosen, rgrad, x), max_iter=50000, critere="grad")
print("pas fixe 1e-3 :", ha["statut"], ha["iters"], xa)
print("backtracking  :", hb["statut"], hb["iters"], xb2)

**Question 7.4.** Que constatez-vous ? Pourquoi cette fonction est-elle difficile pour la descente de gradient ?

**Réponse :** Le pas fixe $10^{-3}$ n'a pas convergé au bout de 20 000 itérations (statut `max_iter`, point encore à environ $3 \times 10^{-4}$ du minimum) et le backtracking en demande environ 13 800. Le minimum se trouve au fond d'une vallée étroite et courbée : la direction de plus forte pente zigzague d'un bord à l'autre, ce qui est un cas extrême de mauvais conditionnement.

### 7.3 Application : régression linéaire
On dispose de $N = 50$ points et on cherche la droite $y = a x + b$ qui minimise $L(a, b) = \frac{1}{N} \sum_i (a x_i + b - y_i)^2$, de gradient $\left(\frac{2}{N}\sum (a x_i + b - y_i) x_i,\; \frac{2}{N}\sum (a x_i + b - y_i)\right)^\top$.

**Exercice 7.5.** Écrire `L(p)` et `grad_L(p)` pour $p = (a, b)$, lancer la descente (pas $0{,}1$, `max_iter=5000`, `tol=1e-8`, critère `"grad"`) depuis $(0, 0)$ et comparer avec `np.polyfit`. Essayer ensuite les pas $0{,}01$, $0{,}5$ et $0{,}9$.

In [ ]:
rng = np.random.default_rng(0)
xd = np.linspace(0, 1, 50)
yd = 2 * xd + 1 + 0.1 * rng.normal(size=50)

def L(p):
    return np.mean((p[0] * xd + p[1] - yd)**2)
def grad_L(p):
    r = p[0] * xd + p[1] - yd
    return np.array([2 * np.mean(r * xd), 2 * np.mean(r)])

p_gd, hL = descente(L, grad_L, np.zeros(2), 0.1, max_iter=5000, tol=1e-8, critere="grad")
print("descente de gradient :", p_gd, hL["iters"], "itérations")
print("np.polyfit           :", np.polyfit(xd, yd, 1))
for t in [0.01, 0.5, 0.9]:
    pt, ht = descente(L, grad_L, np.zeros(2), t, max_iter=3000, tol=1e-8, critere="grad")
    print(f"pas = {t:4.2f} : statut = {ht['statut']:10s}, itérations = {ht['iters']:5d}")

**Question 7.6.** Que représente le pas ici (*learning rate*) ? Quelle limite observe-t-on ?

**Réponse :** Le pas est la vitesse d'apprentissage. $0{,}1$ et $0{,}5$ convergent vers la même droite que `polyfit` ; $0{,}01$ est trop lent (3000 itérations ne suffisent pas) ; $0{,}9$ diverge, car il dépasse $2/\lambda_{\max}$ où $\lambda_{\max} \approx 2{,}5$ est la plus grande valeur propre de la hessienne de $L$ (pas critique d'environ $0{,}8$).

## Partie 8 : synthèse
**Question 8.1.** Vous devez écrire une fonction de descente de gradient pour un collègue qui ne connaît pas la fonction à minimiser à l'avance. Quel critère d'arrêt lui recommandez-vous, avec quelles protections ? Justifiez avec ce que vous avez observé aux parties 3.2 à 3.9.

**Réponse :** Un critère **relatif** sur le gradient (`grad_rel`), qui ne dépend pas de l'échelle de $f$ et qui est directement lié à $\nabla f = 0$, complété par un **nombre maximal d'itérations** et une **détection de divergence**, et en affichant le statut (jamais « convergé » sans le dire). Les critères `pas` et `fval` sont tentants mais peuvent s'arrêter loin du minimum (fausse convergence avec un petit pas, arrêt précoce avec `fval`).

**Question 8.2.** Résumez en quelques lignes comment choisir entre pas fixe, backtracking et pas optimal.

**Réponse :** Pas fixe : simple et peu coûteux par itération, mais il faut connaître (ou estimer) $\lambda_{\max}$ ; trop petit il est lent, trop grand il oscille ou diverge. Backtracking : robuste et général, sans connaissance préalable de la fonction, au prix de plusieurs évaluations de $f$ par itération. Pas optimal : n'a de sens simple que pour une fonction quadratique, où il peut être très bon ou équivalent à un pas fixe selon le point de départ. Dans tous les cas, un mauvais conditionnement ralentit la méthode : c'est ce que corrigent Newton et quasi-Newton (séance 7).

**Question 8.3.** Quelle est, selon vous, l'erreur la plus dangereuse que vous avez rencontrée dans ce TP, et comment l'éviter ?

**Réponse :** La plus dangereuse est la **fausse convergence** : l'algorithme annonce qu'il a convergé alors qu'il s'est arrêté loin du minimum (pas minuscule avec `pas` ou `fval`, seuil absolu sur une fonction à petite échelle). On l'évite en utilisant un critère lié au gradient, relatif, en vérifiant le statut, et en contrôlant que la valeur de $f$ et le gradient obtenus sont cohérents avec un minimum.